## Advanced Rag System

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Navigate to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

load_dotenv()

# 2. Load centralized config & embedding model
from src.services.llm_services import load_config, get_text_embeddings

config = load_config("src/config/config.yaml")
embeddings = get_text_embeddings(config)

print(f"📂 Project Root: {os.getcwd()}")
print(f"✅ Embedding Model: {config['text_emb_provider']} / {config['text_emb_model']}")

c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Project Root: c:\Users\User\Desktop\Operation Legder-Mind
✅ Embedding Model: sbert / sentence-transformers/all-MiniLM-L6-v2


In [2]:
import weaviate
from langchain_weaviate import WeaviateVectorStore

# 1. Connect to local Weaviate running in Docker
client = weaviate.connect_to_local()
INDEX_NAME = "FinancialReportChunks"

if not client.is_ready():
    raise ConnectionError("❌ Local Weaviate is not running. Start Docker container first!")

# 2. Connect to the existing collection (No re-indexing required!)
vectorstore = WeaviateVectorStore(
    client=client,
    index_name=INDEX_NAME,
    text_key="text",
    embedding=embeddings
)

# 3. Check collection count
collection = client.collections.get(INDEX_NAME)
count = collection.aggregate.over_all(total_count=True).total_count

print(f"✅ Connected to existing Weaviate collection '{INDEX_NAME}'")
print(f"📊 Total indexed chunks available: {count}")

simsimd not available — falling back to SciPy/NumPy implementation for vector math. To enable the accelerated path, install the optional dependency 'simsimd' (note: simsimd may require newer glibc on some systems).


✅ Connected to existing Weaviate collection 'FinancialReportChunks'
📊 Total indexed chunks available: 543


In [3]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"

# Perform similarity search
docs = vectorstore.similarity_search(query, k=3)

print("=" * 60)
print(f"🔍 QUERY: {query}")
print("=" * 60)

for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "N/A")
    splitter = doc.metadata.get("splitter", "N/A")
    
    print(f"\n🎯 [Rank #{i}] (SEC 10-K Page: {page} | Strategy: {splitter})")
    print("-" * 60)
    print(doc.page_content.strip())
    print("-" * 60)

🔍 QUERY: What is Uber's core mission statement as described in the 2024 annual report?

🎯 [Rank #1] (SEC 10-K Page: 1.0 | Strategy: recursive)
------------------------------------------------------------
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
what gets us out of bed each morning. It pushes us to constantly reimagine 
how we can move better. For you. For all the places you want to go. For all the 
things you want to get. For all the ways you want to earn. Across the entire 
world. In real time. At the incredible speed of now.
------------------------------------------------------------

🎯 [Rank #2] (SEC 10-K Page: 11.0 | Strategy: recursive)
------------------------------------------------------------
Employees have access to an in

## Hybrid Search(Dense(vector search) +Sparce(BM25))

In [ ]:
from collections import defaultdict
from weaviate.classes.query import HybridFusion
from typing import List

if not client.is_connected():
    client.connect()

def rrf_fusion(dense_results, bm25_results, k=60, top_n=3) -> List:
    """
    Reciprocal Rank Fusion (RRF) - combines dense and sparse retrieval.
    
    RRF Formula: score(d) = Σ 1/(k + rank(d)) for each ranking
    
    - k=60 is the default constant (from original paper)
    - Higher k → more weight to lower-ranked documents
    - Lower k → more weight to top-ranked documents
    
    Args:
        dense_docs: Results from dense (vector) retrieval
        bm25_results: Results from BM25 (sparse) retrieval
        k: RRF constant (default 60, typical range 1-100)
        
    Returns:
        Fused results sorted by RRF score (higher = more relevant)
    """
    rrf_scores  = defaultdict(float)
    doc_map={}

    #1. Score dense vector ranks
    for rank, doc in enumerate(dense_results,1):
        doc_id= doc.metadata.get("page_content_hash", doc.page_content[:50])
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    # 2. Score BM25 keyword ranks
    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc.metadata.get("page_content_hash", doc.page_content[:50])
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)
    
    # 3. Sort by fused RRF score descending
    sorted_docs = sorted(rrf_scores.items(), key=lambda item:item[1], reverse=True)

    # 4. Return top_n fused documents with their score
    fused_results = []
    for doc_id, score in sorted_docs[:top_n]:
        fused_results.append((doc_map[doc_id], score))
    return fused_results



### 1. Execute Manual RRF Hybrid Search

In [ ]:
from langchain_core.documents import Document

query = "What was Uber's GAAP operating income in 2024?"

# 1. Dense Vector Retrieval (Top 10)
dense_docs = vectorstore.similarity_search(query, k=10)

# 2. Sparse BM25 Keyword Retrieval (Top 10)
bm25_raw = collection.query.bm25(query=query, limit=10)
bm25_docs = [
    Document(
        page_content=obj.properties.get("text", ""),
        metadata={
            "page": obj.properties.get("page"),
            "source": obj.properties.get("source"),
            "splitter": obj.properties.get("splitter")
        }
    )
    for obj in bm25_raw.objects
]

# 3. Fuse rankings using your rrf_fusion function (k=60)
fused_results = rrf_fusion(dense_docs, bm25_docs, k=60, top_n=3)

print("=" * 65)
print(f"🔍 MANUAL RRF HYBRID SEARCH: '{query}'")
print("=" * 65)

for rank, (doc, rrf_score) in enumerate(fused_results, 1):
    page = doc.metadata.get("page", "N/A")
    print(f"\n🏆 [Rank #{rank}] | RRF Score: {rrf_score:.5f} | Page: {page}")
    print("-" * 65)
    print(doc.page_content.strip()[:250] + "...")
    print("-" * 65)

### 2. Execute Native Weaviate Hybrid Search (Built-in RRF)

In [ ]:
# Embed query vector for Weaviate's native hybrid search
query_vector = embeddings.embed_query(query)

# Native Weaviate Hybrid Search using Reciprocal Rank Fusion (HybridFusion.RANKED)
native_hybrid_response = collection.query.hybrid(
    query=query,                       # Sparse / BM25 keyword search
    vector=query_vector,               # Dense / SBERT vector search
    alpha=0.5,                         # 0.5 = Balanced dense & sparse
    fusion_type=HybridFusion.RANKED,   # 👈 Weaviate's native RRF fusion
    limit=3,
    return_metadata=["score"]
)

print("=" * 65)
print(f"⚡ NATIVE WEAVIATE HYBRID SEARCH (RRF): '{query}'")
print("=" * 65)

for i, obj in enumerate(native_hybrid_response.objects, 1):
    score = obj.metadata.score if obj.metadata else 0.0
    page = obj.properties.get("page", "N/A")
    print(f"\n🏆 [Rank #{i}] | Native RRF Score: {score:.5f} | Page: {page}")
    print("-" * 65)
    print(obj.properties.get("text", "").strip()[:250] + "...")
    print("-" * 65)

# ============================================================================
# Part 3: "The Librarian" (Advanced RAG System) [25 Marks]
# ============================================================================

### **The Problem:**
Financial documents contain highly specific entities (e.g., *"Form 10-K"*, *"$162.6B"*, *"GAAP operating income"*, *"Uber One"*) that dense vector search alone often overlooks.

### **Pipeline Architecture:**
1. **Vector Database:** Weaviate (`FinancialReportChunks` collection).
2. **Hybrid Search:** Combines **Dense Vector Search (SBERT)** + **BM25 (Keyword Search)**.
3. **Refinement Stage 1:** **Reciprocal Rank Fusion (RRF)** to merge sparse and dense candidate pools.
4. **Refinement Stage 2:** **Cross-Encoder Neural Re-ranking** (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to score true semantic alignment.
5. **Constraint:** Implements `query_librarian(question)` to return authoritative, grounded financial answers with SEC 10-K page citations.

In [ ]:
from collections import defaultdict
from typing import Dict, Any, List
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from src.services.llm_services import get_llm

# 1. Initialize Cross-Encoder Reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Initialize LLM for grounded answer generation
try:
    llm = get_llm(config)
    print(f"✅ LLM initialized for 'The Librarian': {config.get('llm_provider')} / {config.get('llm_model')}")
except Exception as e:
    print(f"⚠️ LLM provider note: {e}")
    llm = None


def rrf_fusion_scores(dense_results: List[Document], bm25_results: List[Document], k: int = 60) -> List[Dict[str, Any]]:
    """
    Reciprocal Rank Fusion (RRF) combining dense and BM25 candidate lists.
    Formula: score(d) = sum(1 / (k + rank(d)))
    """
    rrf_scores = defaultdict(float)
    doc_map = {}

    # Dense vector ranks
    for rank, doc in enumerate(dense_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    # BM25 keyword ranks
    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc.metadata.get("page_content_hash") or hash(doc.page_content)
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    # Sort descending by fused RRF score
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"doc": doc_map[doc_id], "rrf_score": score, "doc_id": doc_id} for doc_id, score in sorted_items]


def query_librarian(
    question: str,
    top_k_retrieval: int = 10,
    top_k_rerank: int = 3,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    'The Librarian' - Advanced Hybrid RAG Pipeline
    
    Executes:
      1. Weaviate Dense Vector Search (SBERT)
      2. Weaviate Sparse BM25 Keyword Search
      3. Reciprocal Rank Fusion (RRF, k=60)
      4. Neural Cross-Encoder Re-ranking
      5. Grounded Financial LLM Answering with SEC 10-K Citations
    
    Args:
        question: User inquiry regarding SEC 10-K filing
        top_k_retrieval: Number of candidates to retrieve per modality (default: 10)
        top_k_rerank: Final passages to retain after Cross-Encoder (default: 3)
        verbose: Whether to print diagnostic execution trace
        
    Returns:
        dict: {
            'question': str,
            'answer': str,
            'sources': List[float/int],
            'reranked_chunks': List[Dict],
            'metrics': Dict
        }
    """
    # 0. Ensure Weaviate client is active
    if not client.is_connected():
        client.connect()
    
    collection = client.collections.get("FinancialReportChunks")

    # ------------------------------------------------------------------------
    # STEP 1: HYBRID RETRIEVAL (Dense + BM25)
    # ------------------------------------------------------------------------
    # A. Dense Vector Search (SBERT)
    query_vector = embeddings.embed_query(question)
    dense_res = collection.query.near_vector(
        near_vector=query_vector,
        limit=top_k_retrieval,
        return_metadata=["distance"]
    )
    dense_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
                "distance": float(obj.metadata.distance) if obj.metadata else 0.0,
            }
        )
        for obj in dense_res.objects
    ]

    # B. Sparse BM25 Keyword Search
    bm25_res = collection.query.bm25(
        query=question,
        limit=top_k_retrieval,
        return_metadata=["score"]
    )
    bm25_docs = [
        Document(
            page_content=obj.properties.get("text", ""),
            metadata={
                "page": obj.properties.get("page"),
                "source": obj.properties.get("source"),
                "splitter": obj.properties.get("splitter"),
                "score": float(obj.metadata.score) if obj.metadata else 0.0,
            }
        )
        for obj in bm25_res.objects
    ]

    # ------------------------------------------------------------------------
    # STEP 2: REFINEMENT 1 - RECIPROCAL RANK FUSION (RRF)
    # ------------------------------------------------------------------------
    rrf_candidates = rrf_fusion_scores(dense_docs, bm25_docs, k=60)
    top_candidates = rrf_candidates[:top_k_retrieval]

    # ------------------------------------------------------------------------
    # STEP 3: REFINEMENT 2 - CROSS-ENCODER RE-RANKING
    # ------------------------------------------------------------------------
    candidate_docs = [item["doc"] for item in top_candidates]
    ce_pairs = [[question, doc.page_content] for doc in candidate_docs]
    ce_scores = reranker.predict(ce_pairs)

    reranked_pool = []
    for item, ce_score in zip(top_candidates, ce_scores):
        reranked_pool.append({
            "doc": item["doc"],
            "rrf_score": item["rrf_score"],
            "cross_encoder_score": float(ce_score),
            "page": item["doc"].metadata.get("page"),
            "source": item["doc"].metadata.get("source"),
        })

    # Sort descending by neural cross-encoder score
    reranked_pool.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    final_passages = reranked_pool[:top_k_rerank]

    # ------------------------------------------------------------------------
    # STEP 4: GROUNDED FINANCIAL LLM GENERATION
    # ------------------------------------------------------------------------
    context_blocks = []
    sources = []
    for p in final_passages:
        page_num = p["page"] if p["page"] is not None else "Unknown"
        sources.append(page_num)
        context_blocks.append(f"[SEC Form 10-K Page {page_num}]:\n{p['doc'].page_content.strip()}")

    formatted_context = "\n\n".join(context_blocks)

    system_instruction = (
        "You are 'The Librarian', an authoritative financial analyst AI specializing in SEC Form 10-K filings. "
        "Answer the question accurately, concisely, and factually based strictly on the provided context. "
        "Always cite the relevant SEC 10-K page numbers for financial metrics and claims. "
        "If the answer cannot be determined directly from the context, state: 'Based on the provided document, this information is not available.'"
    )

    prompt = (
        f"{system_instruction}\n\n"
        f"--- FINANCIAL DOCUMENT EVIDENCE ---\n{formatted_context}\n\n"
        f"--- INQUIRY ---\n{question}\n\n"
        f"--- AUTHORITATIVE ANSWER ---"
    )

    answer = ""
    if llm:
        try:
            response = llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            answer = f"⚠️ LLM generation failed: {e}. (See retrieved passages below)"
    else:
        # Grounded summary from top retrieved chunk if LLM client is not connected
        answer = f"Top Evidence (Page {final_passages[0]['page']}):\n{final_passages[0]['doc'].page_content.strip()[:300]}..."

    # ------------------------------------------------------------------------
    # STEP 5: DIAGNOSTIC TRACE OUTPUT
    # ------------------------------------------------------------------------
    if verbose:
        print("=" * 70)
        print("📚 'THE LIBRARIAN' ADVANCED RAG PIPELINE")
        print("=" * 70)
        print(f"❓ Question: {question}")
        print(f"🔍 Hybrid Search:  {len(dense_docs)} Dense + {len(bm25_docs)} BM25 results")
        print(f"🔀 RRF Refinement: Merged into {len(top_candidates)} candidate chunks")
        print(f"🎯 Cross-Encoder:  Re-ranked and filtered to Top {len(final_passages)} passages")
        print("-" * 70)
        print(f"💡 ANSWER:\n{answer.strip()}\n")
        print("-" * 70)
        print("📑 RE-RANKED CITATIONS:")
        for rank, p in enumerate(final_passages, 1):
            print(f"  [{rank}] Page {p['page']} | CE Score: {p['cross_encoder_score']:+.4f} | RRF Score: {p['rrf_score']:.5f}")
            print(f"      {p['doc'].page_content.strip()[:140]}...")
        print("=" * 70)

    return {
        "question": question,
        "answer": answer.strip(),
        "sources": sources,
        "reranked_chunks": final_passages,
        "metrics": {
            "dense_retrieved": len(dense_docs),
            "bm25_retrieved": len(bm25_docs),
            "rrf_candidates": len(top_candidates),
            "final_passages": len(final_passages)
        }
    }


### 3. Test `query_librarian` on Complex Financial Questions

In [ ]:
# Test 1: Numerical entity & Gross Bookings
result_1 = query_librarian("What was Uber's total Gross Bookings reported for the fiscal year 2024?")

# Test 2: Specific corporate program entity
result_2 = query_librarian("What benefits do Uber One members receive according to the report?")